<a href="https://colab.research.google.com/github/Tejaswinns/Taiwan-Bankrupcy-Project/blob/main/Taiwan_Bankrupcy_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

import os

print(os.listdir("/content/drive/MyDrive"))


['Colab Notebooks', 'Week 3 Assignment(visualization).gsheet', 'Dashboard vision using eye-tracking to understand and predict Dashboard Viewing Behaviors(Research Paper).gdoc', 'Netflix_NLP_Column', 'NCHS_-_Drug_Poisoning_Mortality_by_State__United_States_20250827.xlsx', 'Question 7.drawio.png', 'Question 7.drawio', 'Question 8.drawio.png', 'Module 5.drawio', 'Module 5.drawio.png', 'Doctor Appointments (1).drawio.png', 'Question 4.drawio', 'Question 4.drawio.xml', 'Question 4.drawio.png', 'Question3_Doctor_Appointment .drawio.png', '3NF_Updated.drawio.png', '3NF_Updated_1.drawio.png', '3NF_Updated.drawio', '3NF_Updated_2.drawio.png', '1NF(PROJECT).drawio.png', 'Prenormalization.drawio.pdf', 'Prenormalization-Page-1.drawio.pdf', '1NF.drawio.pdf', '2NF-Page-3.drawio.pdf', 'DATABOOK2-2NF_TABLE2.drawio.pdf', 'DATABOOK2-2NF TABLE_1.drawio.pdf', 'DATABOOK2-2NF_TABLE3.drawio.pdf', 'ERD.drawio.pdf', 'Copy of ERD.drawio.pdf', 'NCHS_-_Childhood_Mortality_Rates_20260407.xlsx', 'Databook_2', 'DATA

In [4]:

import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/Taiwan_Dataset/data.csv"
)

print("Dataset shape:", df.shape)


Dataset shape: (6819, 96)


In [5]:

print("Dataset shape:", df.shape)
print("Total companies:", len(df))
print("Total columns:", len(df.columns))


Dataset shape: (6819, 96)
Total companies: 6819
Total columns: 96


In [6]:

df["Bankrupt?"].value_counts()


,count
Bankrupt?,
0,6599
1,220


In [7]:

print("Total missing values:", df.isnull().sum().sum())


Total missing values: 0


In [8]:

print("Duplicate rows:", df.duplicated().sum())


Duplicate rows: 0


In [9]:

# X contains the 95 financial features
X = df.drop(columns=["Bankrupt?"])

# y contains the bankruptcy outcome
y = df["Bankrupt?"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)


Features shape: (6819, 95)
Target shape: (6819,)


In [10]:

# Splitting the columns
X = df.drop(columns=["Bankrupt?"])
y = df["Bankrupt?"]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (6819, 95)
y shape: (6819,)


In [11]:

import numpy as np

print("Missing values:", X.isna().sum().sum())
print("Negative values:", (X < 0).sum().sum())
print("Zero values:", (X == 0).sum().sum())
print("Minimum value:", X.min().min())
print("Maximum value:", X.max().max())


Missing values: 0
Negative values: 0
Zero values: 14616
Minimum value: 0.0
Maximum value: 10000000000.0


In [12]:

from sklearn.model_selection import RepeatedStratifiedKFold

cv = RepeatedStratifiedKFold(
    n_splits=10,
    n_repeats=3,
    random_state=42
)

print("Total evaluations:", cv.get_n_splits(X, y))


Total evaluations: 30


In [13]:

# Step 1: Count how many zero values are present in each of the 95 financial features.
# (X == 0) checks whether each value is zero.
# .sum() counts the zeros separately for each feature (column).
zero_counts = (X == 0).sum()

# Step 2: Count how many negative values are present in each financial feature.
# (X < 0) checks whether each value is negative.
# .sum() counts the negative values in each feature (column).
negative_counts = (X < 0).sum()

# Step 3: Create a DataFrame to organize the results.
# The DataFrame contains two columns:
# "Zero values" and "Negative values".
# Each row represents one financial feature.
feature_check = pd.DataFrame({
    "Zero values": zero_counts,
    "Negative values": negative_counts
})

# Step 4: Display only the financial features that contain
# at least one zero OR at least one negative value.
# The | symbol means OR.
# Features with no zero or negative values are excluded from the output.
print(feature_check[
    (feature_check["Zero values"] > 0) |
    (feature_check["Negative values"] > 0)
])


                                                    Zero values  \
ROA(C) before interest and depreciation before ...            1   
ROA(A) before interest and % after tax                        1   
ROA(B) before interest and depreciation after tax             1   
Operating Gross Margin                                        1   
Realized Sales Gross Margin                                   1   
...                                                         ...   
Net Income to Stockholder's Equity                            1   
Liability to Equity                                           1   
Degree of Financial Leverage (DFL)                            1   
Interest Coverage Ratio (Interest expense to EBIT)            1   
Equity to Liability                                           1   

                                                    Negative values  
ROA(C) before interest and depreciation before ...                0  
ROA(A) before interest and % after tax                 

In [14]:

# Step 1: Count how many financial features contain at least one zero.
features_with_zeros = (zero_counts > 0).sum()

# Step 2: Count how many financial features contain at least one negative value.
features_with_negatives = (negative_counts > 0).sum()

# Step 3: Count all zero values across the 95 financial features.
total_zeros = zero_counts.sum()

# Step 4: Count all negative values across the 95 financial features.
total_negatives = negative_counts.sum()

# Step 5: Display the results.
print("Features containing zeros:", features_with_zeros)
print("Features containing negatives:", features_with_negatives)
print("Total zero values:", total_zeros)
print("Total negative values:", total_negatives)


Features containing zeros: 94
Features containing negatives: 0
Total zero values: 14616
Total negative values: 0


In [15]:

import numpy as np

# Step 1: Select one financial feature that contains zeros.
# We will use it only to understand the transformation.
feature_name = zero_counts[zero_counts > 0].index[0]

# Step 2: Select one zero value and one positive value
# from that feature for comparison.
zero_value = 0.0
positive_value = X.loc[X[feature_name] > 0, feature_name].iloc[0]

# Step 3: Display the original values.
print("Feature:", feature_name)
print("Original zero:", zero_value)
print("Original positive value:", positive_value)

# Step 4: Examine two possible transformations.
# log(0) is undefined, while log1p(0) is valid.
# This is a diagnostic comparison, NOT the paper's confirmed method.
print("log(positive value):", np.log(positive_value))
print("log1p(zero):", np.log1p(zero_value))
print("log1p(positive value):", np.log1p(positive_value))


Feature:  ROA(C) before interest and depreciation before interest
Original zero: 0.0
Original positive value: 0.370594257300249
log(positive value): -0.9926474609329419
log1p(zero): 0.0
log1p(positive value): 0.31524441024416544


In [16]:

import numpy as np

from sklearn.preprocessing import FunctionTransformer, MinMaxScaler
from sklearn.pipeline import Pipeline

# Step 1: Create a logarithmic transformation.
# log1p(x) calculates log(1 + x).
# This handles zero values without producing undefined results.
#
# IMPORTANT:
# The original paper does not clearly specify how zeros
# were handled. We are using log1p as a documented assumption.
log_transformer = FunctionTransformer(
    np.log1p,
    validate=False
)

# Step 2: Create the preprocessing pipeline.
# First, apply the logarithmic transformation.
# Then, scale each feature to the range 0 to 1.
preprocessing = Pipeline([
    ("log_transform", log_transformer),
    ("minmax_scaler", MinMaxScaler())
])

# Step 3: Confirm that the pipeline was created.
# This does not transform the dataset or train a model yet.
print("Preprocessing pipeline created successfully!")


Preprocessing pipeline created successfully!


In [17]:

from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, fbeta_score
import numpy as np

# Step 1: Create Model A without ENN.
# Apply preprocessing, then train Gaussian Naive Bayes.
model_A = Pipeline([
    ("preprocessing", preprocessing),
    ("naive_bayes", GaussianNB())
])

# Step 2: Define F2-score as our evaluation metric.
# F2 emphasizes recall more than precision.
f2_scorer = make_scorer(
    fbeta_score,
    beta=2,
    zero_division=0
)

# Step 3: Run repeated stratified 10-fold cross-validation.
# 10 folds x 3 repeats = 30 evaluations.
# Preprocessing is fitted separately on each training fold.
scores_A = cross_val_score(
    model_A,
    X,
    y,
    cv=cv,
    scoring=f2_scorer,
    n_jobs=-1
)

# Step 4: Calculate the average score and its variation.
mean_f2_A = np.mean(scores_A)
std_f2_A = np.std(scores_A, ddof=1)

# Step 5: Display the results.
print("Number of evaluations:", len(scores_A))
print("Model A mean F2-score:", round(mean_f2_A, 4))
print("Model A standard deviation:", round(std_f2_A, 4))


Number of evaluations: 30
Model A mean F2-score: 0.4017
Model A standard deviation: 0.0809


In [18]:

# Import the required libraries.
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler
from sklearn.naive_bayes import GaussianNB
from imblearn.under_sampling import EditedNearestNeighbours
from imblearn.pipeline import Pipeline as ImbPipeline
import numpy as np

# Step 1: Create Model B using one pipeline.
# The steps are executed in the order shown below.

model_B = ImbPipeline([

    # Step 2: Apply logarithmic transformation to all 95 features.
    # log1p handles zero values.
    # This is our documented reproduction assumption.
    ("log_transform", FunctionTransformer(np.log1p)),

    # Step 3: Scale the financial features.
    # The scaler learns from the training fold only.
    ("minmax_scaler", MinMaxScaler()),

    # Step 4: Apply ENN to training companies only.
    # Check the 3 nearest neighbors.
    # Remove selected majority-class companies.
    ("enn", EditedNearestNeighbours(
        n_neighbors=3,
        sampling_strategy="majority",
        kind_sel="all"
    )),

    # Step 5: Train Gaussian Naive Bayes
    # using the data remaining after ENN.
    ("naive_bayes", GaussianNB())

])

# Step 6: Display the corrected pipeline.
print(model_B)


Pipeline(steps=[('log_transform', FunctionTransformer(func=<ufunc 'log1p'>)),
                ('minmax_scaler', MinMaxScaler()),
                ('enn', EditedNearestNeighbours(sampling_strategy='majority')),
                ('naive_bayes', GaussianNB())])


In [19]:

from sklearn.model_selection import cross_val_score
import numpy as np

# Step 1: Run Model B using the same 30 evaluations as Model A.
# ENN is applied only during training, never to the test folds.
scores_B = cross_val_score(
    model_B,
    X,
    y,
    cv=cv,
    scoring=f2_scorer,
    n_jobs=1,
    error_score="raise"
)

# Step 2: Calculate the mean F2-score.
mean_f2_B = np.mean(scores_B)

# Step 3: Calculate the standard deviation.
std_f2_B = np.std(scores_B, ddof=1)

# Step 4: Display the results.
print("Number of evaluations:", len(scores_B))
print("Model B mean F2-score:", round(mean_f2_B, 4))
print("Model B standard deviation:", round(std_f2_B, 4))


Number of evaluations: 30
Model B mean F2-score: 0.4289
Model B standard deviation: 0.0824


In [20]:

import numpy as np
import pandas as pd

# Step 1: Calculate the F2-score difference for each
# matching cross-validation evaluation.
# Positive values mean Model B performed better.
f2_differences = scores_B - scores_A

# Step 2: Create a table showing both models' scores
# and their differences for all 30 evaluations.
comparison = pd.DataFrame({
    "Evaluation": range(1, 31),
    "Model A (without ENN)": scores_A,
    "Model B (with ENN)": scores_B,
    "Difference (B - A)": f2_differences
})

# Step 3: Display the first five comparisons.
print(comparison.head())

# Step 4: Summarize the differences.
print("\nAverage F2 improvement:", round(np.mean(f2_differences), 4))
print("Evaluations where ENN improved F2:",
      np.sum(f2_differences > 0))
print("Evaluations where ENN reduced F2:",
      np.sum(f2_differences < 0))
print("Evaluations with no change:",
      np.sum(f2_differences == 0))


   Evaluation  Model A (without ENN)  Model B (with ENN)  Difference (B - A)
0           1               0.389610            0.382166           -0.007445
1           2               0.454545            0.378788           -0.075758
2           3               0.136861            0.147638            0.010776
3           4               0.531915            0.523256           -0.008659
4           5               0.454545            0.555556            0.101010

Average F2 improvement: 0.0272
Evaluations where ENN improved F2: 21
Evaluations where ENN reduced F2: 8
Evaluations with no change: 1


In [21]:

import numpy as np
import pandas as pd

# Step 1: Calculate the difference between Model B and Model A
# for each of the 30 matching cross-validation evaluations.
# Positive difference = ENN performed better.
# Negative difference = ENN performed worse.
f2_differences = scores_B - scores_A

# Step 2: Create a table containing both models' F2-scores
# and the difference for each evaluation.
comparison = pd.DataFrame({
    "Evaluation": range(1, len(scores_A) + 1),
    "Model A (Without ENN)": scores_A,
    "Model B (With ENN)": scores_B,
    "F2 Difference (B - A)": f2_differences
})

# Step 3: Display the first 10 evaluations.
print(comparison.head(10).to_string(index=False))

# Step 4: Count how often ENN improved or reduced the F2-score.
improved = np.sum(f2_differences > 0)
reduced = np.sum(f2_differences < 0)
unchanged = np.sum(f2_differences == 0)

# Step 5: Display the summary.
print("\nAverage F2 improvement:", round(np.mean(f2_differences), 4))
print("Evaluations improved:", improved)
print("Evaluations reduced:", reduced)
print("Evaluations unchanged:", unchanged)


 Evaluation  Model A (Without ENN)  Model B (With ENN)  F2 Difference (B - A)
          1               0.389610            0.382166              -0.007445
          2               0.454545            0.378788              -0.075758
          3               0.136861            0.147638               0.010776
          4               0.531915            0.523256              -0.008659
          5               0.454545            0.555556               0.101010
          6               0.392157            0.451977               0.059821
          7               0.386598            0.406091               0.019493
          8               0.342742            0.410256               0.067514
          9               0.483871            0.460123              -0.023748
         10               0.492228            0.533708               0.041480

Average F2 improvement: 0.0272
Evaluations improved: 21
Evaluations reduced: 8
Evaluations unchanged: 1


In [22]:

import numpy as np
from scipy.stats import t

# Step 1: Calculate the F2-score difference for each
# of the 30 matching evaluations.
# Model B = With ENN
# Model A = Without ENN
differences = scores_B - scores_A

# Step 2: Count the total paired evaluations.
# We have 10 folds repeated 3 times = 30 evaluations.
n = len(differences)

# Step 3: Calculate the average improvement from ENN.
mean_difference = np.mean(differences)

# Step 4: Calculate the variance of the 30 differences.
# This measures how much ENN's improvement varies.
variance_difference = np.var(differences, ddof=1)

# Step 5: Calculate the test-to-training size ratio.
# Each evaluation uses 1 fold for testing
# and 9 folds for training.
test_train_ratio = 1 / 9

# Step 6: Apply the corrected variance formula.
# This approximately accounts for overlapping
# training data across cross-validation evaluations.
corrected_variance = (
    (1 / n) + test_train_ratio
) * variance_difference

# Step 7: Calculate the corrected t-statistic.
corrected_t = mean_difference / np.sqrt(corrected_variance)

# Step 8: Calculate the two-sided p-value.
# We use n - 1 degrees of freedom.
p_value = 2 * t.sf(abs(corrected_t), df=n - 1)

# Step 9: Calculate a 95% confidence interval.
standard_error = np.sqrt(corrected_variance)
critical_value = t.ppf(0.975, df=n - 1)

ci_lower = mean_difference - critical_value * standard_error
ci_upper = mean_difference + critical_value * standard_error

# Step 10: Display the statistical test results.
print("Mean F2 improvement:", round(mean_difference, 4))
print("Corrected t-statistic:", round(corrected_t, 4))
print("Two-sided p-value:", round(p_value, 4))
print("95% confidence interval:",
      (round(ci_lower, 4), round(ci_upper, 4)))


Mean F2 improvement: 0.0272
Corrected t-statistic: 1.5025
Two-sided p-value: 0.1438
95% confidence interval: (np.float64(-0.0098), np.float64(0.0642))


In [23]:

from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, fbeta_score
import numpy as np

# Step 1: Define the performance measurements.
# Precision: How often a bankruptcy prediction is correct.
# Recall: How many actual bankrupt companies are detected.
# F1: Combines precision and recall.
# F2: Gives more importance to recall.

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "f2": make_scorer(fbeta_score, beta=2, zero_division=0)
}

# Step 2: Evaluate Model A using the same 30 splits.
# Model A does not use ENN.
results_A = cross_validate(
    model_A,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Step 3: Evaluate Model B using the same 30 splits.
# Model B applies ENN only to the training folds.
results_B = cross_validate(
    model_B,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Step 4: Display average performance across 30 evaluations.
for metric in scoring:
    print(
        metric,
        "| Without ENN:",
        round(np.mean(results_A["test_" + metric]), 4),
        "| With ENN:",
        round(np.mean(results_B["test_" + metric]), 4)
    )


precision | Without ENN: 0.1471 | With ENN: 0.1796
recall | Without ENN: 0.7606 | With ENN: 0.7015
f1 | Without ENN: 0.2423 | With ENN: 0.2802
f2 | Without ENN: 0.4017 | With ENN: 0.4289


In [24]:

import numpy as np
from sklearn.base import clone
from sklearn.metrics import confusion_matrix

# Step 1: Create empty confusion matrices.
# Rows represent actual classes.
# Columns represent predicted classes.
cm_A = np.zeros((2, 2), dtype=int)
cm_B = np.zeros((2, 2), dtype=int)

# Step 2: Repeat the same 30 cross-validation evaluations.
for train_idx, test_idx in cv.split(X, y):

    # Select training and testing companies.
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Step 3: Train fresh copies of both models.
    # This keeps each evaluation independent of previous fits.
    fitted_A = clone(model_A)
    fitted_B = clone(model_B)

    fitted_A.fit(X_train, y_train)
    fitted_B.fit(X_train, y_train)

    # Step 4: Predict the held-out companies.
    pred_A = fitted_A.predict(X_test)
    pred_B = fitted_B.predict(X_test)

    # Step 5: Add the confusion matrix counts
    # from this evaluation.
    cm_A += confusion_matrix(y_test, pred_A, labels=[0, 1])
    cm_B += confusion_matrix(y_test, pred_B, labels=[0, 1])

# Step 6: Display the combined results.
print("Model A — Without ENN")
print(cm_A)

print("\nModel B — With ENN")
print(cm_B)


Model A — Without ENN
[[16321  3476]
 [  158   502]]

Model B — With ENN
[[17201  2596]
 [  197   463]]


In [25]:

from sklearn.base import clone
import joblib
import os

# Step 1: Recreate the same 30 cross-validation splits.
# We use the same cv settings as our baseline experiment.
splits = list(cv.split(X, y))

# Step 2: Select the first cross-validation fold.
# Both models will use the same training and testing companies.
train_idx, test_idx = splits[0]

# Step 3: Separate the training and testing companies.
X_train_fold = X.iloc[train_idx]
X_test_fold = X.iloc[test_idx]

y_train_fold = y.iloc[train_idx]
y_test_fold = y.iloc[test_idx]

# Step 4: Create fresh copies of Model A and Model B.
# Model A = Naive Bayes without ENN.
# Model B = Naive Bayes with ENN.
saved_model_A = clone(model_A)
saved_model_B = clone(model_B)

# Step 5: Train both models on the same training companies.
# Model B applies ENN only during training.
saved_model_A.fit(X_train_fold, y_train_fold)
saved_model_B.fit(X_train_fold, y_train_fold)

# Step 6: Create a folder in Google Drive.
# This folder will store our trained models and test data.
save_folder = "/content/drive/MyDrive/Taiwan_Dataset/baseline_models"
os.makedirs(save_folder, exist_ok=True)

# Step 7: Save the two fitted models.
joblib.dump(saved_model_A, f"{save_folder}/model_A_fold1.joblib")
joblib.dump(saved_model_B, f"{save_folder}/model_B_fold1.joblib")

# Step 8: Save the matching held-out test companies.
# Both models will be explained using these same companies.
X_test_fold.to_csv(f"{save_folder}/X_test_fold1.csv")
y_test_fold.to_csv(f"{save_folder}/y_test_fold1.csv")

# Step 9: Save the training and testing row indices.
# This allows us to reconstruct the same fold later.
joblib.dump(
    {"train_idx": train_idx, "test_idx": test_idx},
    f"{save_folder}/fold1_indices.joblib"
)

# Step 10: Confirm everything was saved.
print("Models and test data saved successfully!")
print("Training companies:", len(X_train_fold))
print("Testing companies:", len(X_test_fold))


Models and test data saved successfully!
Training companies: 6137
Testing companies: 682


In [26]:

import os

# Display all files saved in the baseline_models folder.
print(os.listdir(save_folder))


['model_A_fold1.joblib', 'model_B_fold1.joblib', 'X_test_fold1.csv', 'y_test_fold1.csv', 'fold1_indices.joblib']
